## Inference Notebook

In this notebook, we will demonstrate how to perform inference using a custom model registered in our model registry. To illustrate the process, we will generate synthetic data, which will serve as the basis for our forecasting tasks.

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
session = get_active_session()
## get prophet model
from prophet import Prophet
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model
from snowflake.ml.model import model_signature

import seaborn as sns
import matplotlib.pyplot as plt


The funtion create_datedf creates data for forecasting the timeseries. we will pass the max_date parameter in the notebook


In [ ]:
from datetime import date
today = date.today()
print(today)

db ='ML_MODELS'
schema= 'DS'
warehouse = 'ML_FS_WH'

## function to create data
def create_datedf(max_date, period=7): 
    # Create a new DataFrame starting from max_date + 1 day, adding 7 days to each subsequent row
    date_range = pd.date_range(start=max_date + pd.Timedelta(days=1), periods=period)
    new_df = pd.DataFrame({'ds': date_range})
    return new_df


forecast_dates = create_datedf(today)
forecast_dates.head() 

In [ ]:
reg = Registry(session,database_name = db,schema_name= schema)
model_name='PROPHET_FORCAST_MODEL'
version = 'V1'
mv = reg.get_model('PROPHET_FORCAST_MODEL').version('VERSION_2')

predicted_sales = mv.run(forecast_dates)

In [ ]:
predicted_sales['Created_date']=today
predicted_sales.head()


In [ ]:

sdf = session.createDataFrame(predicted_sales)
sdf.write.mode("append").save_as_table("my_forecast_sales", table_type="transient")

In [ ]:
select * from my_forecast_sales

In [ ]:
forecast = predicted_sales
# Plot the forecasted data (predicted values)
plt.plot(forecast['ds'], forecast['yhat'], color='orange', label='Forecasted Sales')  # Red line for forecast

# Optionally, add confidence intervals (forecast uncertainty)
plt.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'], color='gray', alpha=0.3, label='Confidence Interval')

# Add labels and title
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Sales Forecast vs Original Sales')
plt.legend()

# Show the plot
plt.xticks(rotation=45)
plt.show()

In [ ]:
ls @ML_MODELS.DS.MODEL_STG;
selectt * from my_forecast_sales;

In [ ]:
plt.savefig(f"forecasted_sales_{today}.png")
session.file.put("forecasted_sales.png", "@ML_MODELS.DS.MODEL_STG",auto_compress=False)